# Replacing a link instead of changing it

Can an existing link have its variant changed?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/09_specialization_frozen.py`.

The record holds common information; the pressing classes hold details specific to one kind of pressing. A `Specialization` link goes from the common record to those details, and a `Generalization` link points back. The `Classifier` marker declares which data code belongs to which kind. The [relations chapter](../../docs/tutorials/step-21-relations.md#generalization-and-specialization) explains the complete preparation.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## MusicDomain

Group the music catalogue declarations. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


## VinylRecordEntity

Describe the information shared by all pressings. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordEntity(BaseEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    media: str = Field(description="Pressing code")
    pressing: Annotated[
        Specialization[FirstPressEntity | RepressEntity | TestPressEntity],
        Classifier(field="media", codes=Literal["first", "repress", "test"]),
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


## FirstPressEntity

Describe the stamper used for a first pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="First pressing", domain=MusicDomain)
class FirstPressEntity(BaseEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="Pressing identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## RepressEntity

Describe when an album was pressed again. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [5]:
@entity(description="Re-pressing", domain=MusicDomain)
class RepressEntity(BaseEntity):
    """Describe when an album was pressed again."""

    id: str = Field(description="Pressing identifier")
    year: int = Field(description="Re-pressing year")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["repress"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## TestPressEntity

Describe who approved a test pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [6]:
@entity(description="Test pressing", domain=MusicDomain)
class TestPressEntity(BaseEntity):
    """Describe who approved a test pressing."""

    id: str = Field(description="Pressing identifier")
    approved_by: str = Field(description="Reviewer name")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["test"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [7]:
for model in (VinylRecordEntity, FirstPressEntity, RepressEntity, TestPressEntity):
    model.model_rebuild()


## The experiment

Can an existing link have its variant changed?

In [8]:
from pydantic import ValidationError

link = Specialization[FirstPressEntity | RepressEntity | TestPressEntity](
    id="press-1",
    variant="first",
)
try:
    link.variant = "repress"
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    raise AssertionError("A frozen link was changed")
print(link.variant)


frozen_instance
first


## Why this result occurs

The frozen_instance error means assignment was refused. The original code remains first. Construct a new link when representing a different relation; Generalization is frozen in the same way.